In [1]:
# ================= Cell 1 =================
import os
import glob
import json
from docling.document_converter import DocumentConverter
from docling.chunking import HierarchicalChunker

MD_FOLDER = "markdown_outputs"
LOCAL_CHUNKS_FILE = "processed_chunks_per_paper.json"

# Check if chunks are already processed and saved locally
if os.path.exists(LOCAL_CHUNKS_FILE):
    print(f"Loading existing chunks from local file: {LOCAL_CHUNKS_FILE}")
    with open(LOCAL_CHUNKS_FILE, "r", encoding="utf-8") as f:
        paper_chunks_dict = json.load(f)
    print(f"Successfully loaded chunks for {len(paper_chunks_dict)} papers.")
else:
    print(f"Local chunks file not found. Starting Docling chunking process...")
    md_files = glob.glob(os.path.join(MD_FOLDER, "**", "*.md"), recursive=True)
    print(f"Found {len(md_files)} Markdown files.")

    converter = DocumentConverter()
    chunker = HierarchicalChunker()
    
    # Dictionary to store chunks per file: { "filename.md": ["chunk1", "chunk2", ...] }
    paper_chunks_dict = {}

    for file_path in md_files:
        filename = os.path.basename(file_path)
        paper_chunks_dict[filename] = []
        try:
            # Parse the MD file
            doc_result = converter.convert(file_path)
            # Use Docling for hierarchical chunking
            chunks = chunker.chunk(doc_result.document)
            
            for chunk in chunks:
                text = chunk.text.strip()
                if text:
                    paper_chunks_dict[filename].append(text)
                    
            print(f"Processed {filename}: {len(paper_chunks_dict[filename])} chunks generated.")
        except Exception as e:
            print(f"Error processing {filename}: {e}")

    # Save the dictionary locally to avoid running this again
    with open(LOCAL_CHUNKS_FILE, "w", encoding="utf-8") as f:
        json.dump(paper_chunks_dict, f, ensure_ascii=False, indent=4)
    print(f"\nAll chunks saved locally to {LOCAL_CHUNKS_FILE}.")

/home/ying/anaconda3/envs/llm_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading existing chunks from local file: processed_chunks_per_paper.json
Successfully loaded chunks for 8 papers.


In [2]:
# ================= Cell 2 =================
import torch
import json
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F

QUERY = "Analyze the text and extract the main research objective and the key experimental findings or conclusions."

print("Loading Embedding model...")
embed_model_id = "sentence-transformers/all-MiniLM-L6-v2"
embed_tokenizer = AutoTokenizer.from_pretrained(embed_model_id)
embed_model = AutoModel.from_pretrained(embed_model_id)

# Define Mean Pooling function for embeddings
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

# 1. Vectorize the query (only needs to be done once)
encoded_query = embed_tokenizer([QUERY], padding=True, truncation=True, return_tensors='pt')
with torch.no_grad():
    query_output = embed_model(**encoded_query)
query_embedding = mean_pooling(query_output, encoded_query['attention_mask'])
query_embedding = F.normalize(query_embedding, p=2, dim=1)

# Dictionary to store the top retrieved chunks for each paper
top_chunks_per_paper = {}

print("\nRetrieving Top-3 chunks for EACH paper...")
for paper_name, chunks in paper_chunks_dict.items():
    if not chunks:
        print(f"Skipping {paper_name} (No chunks found).")
        continue

    # Vectorize all chunks for the CURRENT paper
    encoded_chunks = embed_tokenizer(chunks, padding=True, truncation=True, return_tensors='pt', max_length=512)
    with torch.no_grad():
        model_output = embed_model(**encoded_chunks)
    chunk_embeddings = mean_pooling(model_output, encoded_chunks['attention_mask'])
    chunk_embeddings = F.normalize(chunk_embeddings, p=2, dim=1)

    # Calculate cosine similarity
    cos_scores = torch.mm(query_embedding, chunk_embeddings.transpose(0, 1))[0]
    
    # Handle cases where a paper might have fewer than 3 chunks in total
    k = min(3, len(cos_scores))
    top_indices = torch.topk(cos_scores, k=k).indices.tolist()

    top_chunks_per_paper[paper_name] = [chunks[idx] for idx in top_indices]
    print(f"Retrieved top {k} chunks for {paper_name}.")

print("\nRetrieval complete.")

Loading Embedding model...

Retrieving Top-3 chunks for EACH paper...
Retrieved top 3 chunks for paper2.md.
Retrieved top 3 chunks for paper26.md.
Retrieved top 3 chunks for paper5.md.
Retrieved top 3 chunks for paper23.md.
Retrieved top 3 chunks for paper4.md.
Retrieved top 3 chunks for paper22.md.
Retrieved top 3 chunks for paper25.md.
Retrieved top 3 chunks for paper7.md.

Retrieval complete.


In [3]:
# ================= Cell 3 =================

# List to store the final prompt messages for each paper
llm_tasks = []

system_prompt = "You are an expert additive manufacturing researcher. Your task is to extract key experimental findings strictly based on the provided text."

for paper_name, retrieved_chunks in top_chunks_per_paper.items():
    # 1. Concatenate the retrieved chunks for this specific paper
    context_text = ""
    for i, chunk in enumerate(retrieved_chunks):
        context_text += f"\n--- Excerpt {i+1} ---\n{chunk}\n"

    # 2. Build the user prompt
    user_content = f"""
[TASK]
Review the context below and extract the key findings for Ti64 fabrication. 

[CONTEXT]
{context_text}

[QUERY]
{QUERY}

[RESPONSE FORMAT]
Output your findings strictly in JSON format without any markdown blocks, explanations, or conversational text:
{{
  "Research_Objective": "A one-sentence summary of what the study aims to achieve",
  "Key_Findings": [
      "Finding 1",
      "Finding 2"
  ]
}}
"""
    # 3. Store the structured messages for LLaMA chat template
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content},
    ]
    
    llm_tasks.append({
        "paper_name": paper_name,
        "messages": messages
    })

print(f"Successfully constructed prompts for {len(llm_tasks)} papers.")
# Print an example of the first task
if llm_tasks:
    print("\n--- Example Prompt for the first paper ---")
    print(llm_tasks[0]["messages"][1]["content"][:500] + "...\n(Truncated for display)")

Successfully constructed prompts for 8 papers.

--- Example Prompt for the first paper ---

[TASK]
Review the context below and extract the key findings for Ti64 fabrication. 

[CONTEXT]

--- Excerpt 1 ---
Research was sponsored by the Army Research Laboratory and was accomplished under Cooperative Agreement Number W911NF-15-20025. The views and conclusions contained in this document are those of the authors and should not be interpreted as representing the official policies, either expressed or implied, of the Army Research Laboratory or the U.S. Government. The U.S. Government is au...
(Truncated for display)


In [ ]:
# ================= Cell 4 =================
import torch
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. Login to Hugging Face (Replace with your actual Token)
HF_TOKEN = "##YOUR_TOKEN" 
login(token=HF_TOKEN)

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

print("\nLoading LLaMA-3.1-8B-Instruct model (This requires significant VRAM)...")
llama_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Load model in bfloat16 to save memory, auto-mapping to available GPUs
llama_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,  
    device_map="auto"            
)

print("\nModel loaded successfully. Starting inference for each paper...\n")
print("="*60)

# 2. Loop through each paper's task and generate the response
for task in llm_tasks:
    paper_name = task["paper_name"]
    messages = task["messages"]
    
    print(f"--> Analyzing: {paper_name}")
    
    # Apply LLaMA-3.1 specific chat template
    input_ids = llama_tokenizer.apply_chat_template(
        messages, 
        add_generation_prompt=True, 
        return_tensors="pt"
    ).to(llama_model.device)

    # Generate output
    with torch.no_grad():
        outputs = llama_model.generate(
            input_ids,
            max_new_tokens=150,     
            temperature=0.1,        # Low temperature for factual extraction
            do_sample=True,
            pad_token_id=llama_tokenizer.eos_token_id
        )

    # Decode and extract ONLY the generated response (remove input prompt)
    response_ids = outputs[0][input_ids.shape[-1]:]
    response_text = llama_tokenizer.decode(response_ids, skip_special_tokens=True)

    print(f"Result for {paper_name}:")
    print(response_text.strip())
    print("-" * 40)

print("\nAll tasks completed successfully!")


Loading LLaMA-3.1-8B-Instruct model (This requires significant VRAM)...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:01<00:00,  3.55it/s]
Some parameters are on the meta device because they were offloaded to the cpu.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Model loaded successfully. Starting inference for each paper...

--> Analyzing: paper2.md
Result for paper2.md:
{
  "Research_Objective": "The study aims to achieve the fabrication of Ti64 using additive manufacturing, but the specific objective is not explicitly stated in the provided text.",
  "Key_Findings": [
    "The specific experimental findings or conclusions for Ti64 fabrication are not explicitly stated in the provided text.",
    "The text mentions the use of m-APO (multi-objective optimization) for demonstrating the objective space and Pareto optimal solutions, but it does not provide specific findings for Ti64 fabrication."
  ]
}
----------------------------------------
--> Analyzing: paper26.md
Result for paper26.md:
{
  "Research_Objective": "The study aims to investigate the fabrication of Ti64 alloy using additive manufacturing techniques.",
  "Key_Findings": [
    "The effects of strain-rate sensitivity on the microstructure of Ti64 alloy were identified.",
    "Micr